In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

---
## Question 10 

In [ ]:
# ── Q10(a): Monthly simple returns using NumPy slicing ──────────────────────
prices = np.array([
    [100, 108, 103, 115, 110, 119, 125, 121, 130, 127, 135, 140],  # Stock A
    [200, 195, 210, 205, 220, 215, 225, 230, 222, 235, 240, 238]   # Stock B
])

# Simple returns: R_t = (P_t - P_{t-1}) / P_{t-1}
# prices[:, 1:] are months 2-12, prices[:, :-1] are months 1-11
returns = (prices[:, 1:] - prices[:, :-1]) / prices[:, :-1]  # shape: (2, 11)

print("Q10(a): Monthly Simple Returns (shape:", returns.shape, ")")
print("Stock A:", np.round(returns[0], 4))
print("Stock B:", np.round(returns[1], 4))

In [ ]:
# ── Q10(b): Annualised mean return and annualised standard deviation ─────────
monthly_mean = returns.mean(axis=1)          # mean across the 11 monthly returns
monthly_std  = returns.std(axis=1, ddof=1)   # sample std (ddof=1)

ann_mean = monthly_mean * 12                 # annualise mean: multiply by 12
ann_std  = monthly_std  * np.sqrt(12)        # annualise std:  multiply by sqrt(12)

print("Q10(b): Annualised Statistics")
for i, name in enumerate(['Stock A', 'Stock B']):
    print(f"  {name}: Annualised Mean = {ann_mean[i]:.4f} ({ann_mean[i]*100:.2f}%),"
          f" Annualised Std = {ann_std[i]:.4f} ({ann_std[i]*100:.2f}%)")

In [ ]:
# ── Q10(c): 2×2 sample covariance matrix and cross-check ────────────────────
cov_matrix = np.cov(returns)   # np.cov treats each row as a variable
print("Q10(c): Sample Covariance Matrix:")
print(np.round(cov_matrix, 6))

# Confirm off-diagonal = rho * sigma_A * sigma_B
rho_AB   = np.corrcoef(returns)[0, 1]
sig_A    = monthly_std[0]
sig_B    = monthly_std[1]
cov_check = rho_AB * sig_A * sig_B

print(f"\n  rho_AB = {rho_AB:.6f}")
print(f"  sigma_A = {sig_A:.6f},  sigma_B = {sig_B:.6f}")
print(f"  rho * sigma_A * sigma_B  = {cov_check:.6f}")
print(f"  np.cov off-diagonal entry = {cov_matrix[0,1]:.6f}")
print(f"  Match: {np.isclose(cov_check, cov_matrix[0,1])}")

---
## Question 11 

In [ ]:
# ── Q11(a): Define mu and Sigma; equal-weight portfolio stats ────────────────
# Parameters from Q6
mu_assets = np.array([0.15, 0.08, 0.05])  # expected returns

sigma = np.array([0.25, 0.12, 0.04])      # individual std deviations
rho12, rho13, rho23 = 0.4, 0.1, 0.2

# Covariance matrix: Sigma_ij = rho_ij * sigma_i * sigma_j
Sigma = np.array([
    [sigma[0]**2,              rho12*sigma[0]*sigma[1], rho13*sigma[0]*sigma[2]],
    [rho12*sigma[0]*sigma[1],  sigma[1]**2,             rho23*sigma[1]*sigma[2]],
    [rho13*sigma[0]*sigma[2],  rho23*sigma[1]*sigma[2], sigma[2]**2            ]
])

print("Q11(a): Covariance Matrix Sigma:")
print(np.round(Sigma, 6))

# Equal-weight portfolio
w_eq = np.array([1/3, 1/3, 1/3])

E_Rp_eq   = w_eq @ mu_assets           # portfolio expected return
var_p_eq  = w_eq @ Sigma @ w_eq        # portfolio variance
sigma_p_eq = np.sqrt(var_p_eq)         # portfolio std

print(f"\nEqual-weight portfolio:")
print(f"  E[Rp] = {E_Rp_eq:.4f} ({E_Rp_eq*100:.2f}%)")
print(f"  Var(p) = {var_p_eq:.6f}")
print(f"  σ_p   = {sigma_p_eq:.4f} ({sigma_p_eq*100:.2f}%)")

In [ ]:
# ── Q11(b): 10 000 random Dirichlet weight vectors ──────────────────────────
np.random.seed(42)
N = 10_000

# Dirichlet with alpha=[1,1,1] gives uniform random weights summing to 1
weights_rand = np.random.dirichlet(np.ones(3), size=N)  # shape: (10000, 3)

# Vectorised: E[Rp] for all portfolios at once
E_Rp_all = weights_rand @ mu_assets           # shape: (10000,)

# Vectorised: sigma_p for all portfolios at once
# Var_p[i] = w[i] @ Sigma @ w[i]  =  (weights_rand @ Sigma) row-wise dot weights_rand
var_p_all   = np.einsum('ij,jk,ik->i', weights_rand, Sigma, weights_rand)  # shape: (10000,)
sigma_p_all = np.sqrt(var_p_all)              # shape: (10000,)

print("Q11(b): Random portfolio simulation complete.")
print(f"  E[Rp] range: [{E_Rp_all.min()*100:.2f}%, {E_Rp_all.max()*100:.2f}%]")
print(f"  σ_p   range: [{sigma_p_all.min()*100:.2f}%, {sigma_p_all.max()*100:.2f}%]")

In [ ]:
# ── Q11(c): Sharpe Ratios for all 10 000 portfolios (vectorised, no loop) ───
Rf = 0.04  # risk-free rate 4%

sharpe_all = (E_Rp_all - Rf) / sigma_p_all   # shape: (10000,)

max_sharpe_idx = np.argmax(sharpe_all)
max_sharpe     = sharpe_all[max_sharpe_idx]
best_weights   = weights_rand[max_sharpe_idx]

print("Q11(c): Sharpe Ratio Results")
print(f"  Maximum Sharpe Ratio : {max_sharpe:.4f}")
print(f"  Best weights         : Asset1={best_weights[0]:.4f},"
      f" Asset2={best_weights[1]:.4f}, Asset3={best_weights[2]:.4f}")
print(f"  Portfolio E[Rp]      : {E_Rp_all[max_sharpe_idx]*100:.2f}%")
print(f"  Portfolio σ_p        : {sigma_p_all[max_sharpe_idx]*100:.2f}%")

---
## Question 12

In [ ]:
# ── Q12(a): σ_p as a function of rho using NumPy (no loops) ─────────────────
mu1, sig1, mu2, sig2 = 0.12, 0.20, 0.06, 0.10
w1, w2 = 0.6, 0.4

rho_vals = np.linspace(-1, 1, 200)  # 200 correlation values in [-1, +1]

# σ²_p = w1²σ1² + w2²σ2² + 2*w1*w2*ρ*σ1*σ2  (all vectorised over rho_vals)
var_p_rho   = (w1**2 * sig1**2
             + w2**2 * sig2**2
             + 2 * w1 * w2 * rho_vals * sig1 * sig2)  # shape: (200,)
sigma_p_rho = np.sqrt(var_p_rho)                       # shape: (200,)

print("Q12(a): σ_p array computed for 200 correlation values.")
print(f"  Shape: {sigma_p_rho.shape}")
print(f"  σ_p at ρ=-1 : {sigma_p_rho[0]*100:.4f}%")
print(f"  σ_p at ρ= 0 : {sigma_p_rho[99]*100:.4f}%")
print(f"  σ_p at ρ=+1 : {sigma_p_rho[-1]*100:.4f}%")

In [ ]:
# ── Q12(b): Find rho at which σ_p is minimised ───────────────────────────────
min_sigma_idx = np.argmin(sigma_p_rho)
rho_at_min    = rho_vals[min_sigma_idx]
min_sigma_p   = sigma_p_rho[min_sigma_idx]

print("Q12(b): Minimum σ_p Analysis")
print(f"  ρ at minimum σ_p : {rho_at_min:.4f}")
print(f"  Minimum σ_p      : {min_sigma_p*100:.4f}%")

In [ ]:
# ── Q12(c): Analytical verification — dσ²_p/dρ = 2*w1*w2*σ1*σ2 > 0 ──────────
#
# σ²_p(ρ) = w1²σ1² + w2²σ2² + 2*w1*w2*ρ*σ1*σ2
#
# d(σ²_p)/dρ = 2 * w1 * w2 * σ1 * σ2
#
d_var_d_rho = 2 * w1 * w2 * sig1 * sig2

print("Q12(c): Analytical Derivative")
print(f"  d(σ²_p)/dρ = 2 * w1 * w2 * σ1 * σ2 = {d_var_d_rho:.6f}")
print(f"  Since w1, w2, σ1, σ2 > 0, the derivative is strictly POSITIVE.")
print(f"  This means σ²_p (and hence σ_p) is INCREASING in ρ.")
print(f"  Therefore the minimum always occurs at ρ = -1 (the left boundary).")
print(f"  σ_p at ρ = -1: |w1*σ1 - w2*σ2| = |{w1}*{sig1} - {w2}*{sig2}|"
      f" = {abs(w1*sig1 - w2*sig2)*100:.2f}%")

---
## Question 13 

In [ ]:
# ── Setup: Simulate weekly prices (given skeleton) ───────────────────────────
np.random.seed(0)
dates        = pd.date_range('2023-01-02', periods=52, freq='W-MON')
mu_weekly    = np.array([0.003, 0.002, 0.001, 0.0015])
sig_weekly   = np.array([0.04,  0.03,  0.02,  0.025])
returns_sim  = np.random.normal(mu_weekly, sig_weekly, (52, 4))
prices_sim   = 100 * np.cumprod(1 + returns_sim, axis=0)
df           = pd.DataFrame(prices_sim, index=dates,
                             columns=['AAPL', 'MSFT', 'GOOGL', 'AMZN'])

print("Price DataFrame shape:", df.shape)
df.head(3)

In [ ]:
# ── Q13(a): Weekly simple returns ────────────────────────────────────────────
df_returns = df.pct_change().dropna()

print("Q13(a): First 3 rows of weekly returns DataFrame:")
print(df_returns.head(3).round(6))
print(f"\n  Shape of returns DataFrame: {df_returns.shape}")
# shape is (51, 4) — 51 weekly returns from 52 price observations

In [ ]:
# ── Q13(b): Summary statistics ────────────────────────────────────────────────
desc = df_returns.describe()
print("Q13(b): Summary Statistics of Weekly Returns:")
print(desc.round(6))

highest_mean = desc.loc['mean'].idxmax()
highest_std  = desc.loc['std'].idxmax()
print(f"\n  Highest mean return : {highest_mean} ({desc.loc['mean', highest_mean]*100:.4f}%)")
print(f"  Highest std dev     : {highest_std}  ({desc.loc['std',  highest_std ]*100:.4f}%)")

In [ ]:
# ── Q13(c): Annualised Sharpe Ratio for each asset (Rf = 2% annualised) ──────
Rf_annual = 0.02
Rf_weekly = Rf_annual / 52            # weekly risk-free rate

# Annualised mean and std using Pandas (no loops)
ann_mean_pd = df_returns.mean() * 52                  # annualise by 52 weeks
ann_std_pd  = df_returns.std()  * np.sqrt(52)         # annualise by sqrt(52)

sharpe_annual = (ann_mean_pd - Rf_annual) / ann_std_pd

print("Q13(c): Annualised Sharpe Ratios:")
print(sharpe_annual.round(4).to_string())

---
## Question 14

In [ ]:
# ── Q14(a): 4×4 Correlation matrix ───────────────────────────────────────────
corr_matrix = df_returns.corr()
print("Q14(a): Correlation Matrix:")
print(corr_matrix.round(4))

# Find the pair with lowest correlation (excluding self-correlations)
corr_arr = corr_matrix.to_numpy().copy()
np.fill_diagonal(corr_arr, np.nan)
corr_no_diag = pd.DataFrame(corr_arr, index=corr_matrix.index, columns=corr_matrix.columns)

min_corr_val = corr_no_diag.stack().min()
min_corr_pair = corr_no_diag.stack().idxmin()
print(f"\n  Lowest correlation pair: {min_corr_pair[0]} & {min_corr_pair[1]}"
      f"  (ρ = {min_corr_val:.4f})")

In [ ]:
# ── Q14(b): Equal-weight portfolio return series ──────────────────────────────
weights_eq = pd.Series({'AAPL': 0.25, 'MSFT': 0.25, 'GOOGL': 0.25, 'AMZN': 0.25})

# Portfolio return each week = weighted sum of asset returns
portfolio_returns = df_returns.dot(weights_eq)  # Pandas dot product

print("Q14(b): Equal-Weight Portfolio Return Series (first 5):")
print(portfolio_returns.head().round(6))
print(f"\n  Shape: {portfolio_returns.shape}")

In [ ]:
# ── Q14(c): Resample to monthly returns ──────────────────────────────────────
# Monthly return from weekly returns: compound them => (1+r1)*(1+r2)*...-1
monthly_port_returns = portfolio_returns.resample('ME').apply(
    lambda x: (1 + x).prod() - 1
)

print("Q14(c): Monthly Portfolio Return Series:")
print(monthly_port_returns.round(4))
print(f"\n  Mean monthly return : {monthly_port_returns.mean()*100:.4f}%")
print(f"  Std  monthly return : {monthly_port_returns.std()*100:.4f}%")

---
## Question 15 

In [ ]:
# ── Q15 Setup: Simulate 20 000 portfolios for the 3-asset universe ────────────
np.random.seed(42)
N15 = 20_000

# Reuse mu_assets and Sigma from Q11
weights_20k   = np.random.dirichlet(np.ones(3), size=N15)  # (20000, 3)
E_Rp_20k      = weights_20k @ mu_assets                    # (20000,)
var_p_20k     = np.einsum('ij,jk,ik->i', weights_20k, Sigma, weights_20k)
sigma_p_20k   = np.sqrt(var_p_20k)                         # (20000,)
sharpe_20k    = (E_Rp_20k - Rf) / sigma_p_20k              # (20000,)

# Max Sharpe portfolio
msr_idx    = np.argmax(sharpe_20k)
msr_sigma  = sigma_p_20k[msr_idx]
msr_ret    = E_Rp_20k[msr_idx]

# Two-asset Q12 data (already computed above)
weighted_avg_risk = w1 * sig1 + w2 * sig2  # benchmark line

print("Setup complete.")
print(f"  Max Sharpe portfolio: E[Rp]={msr_ret*100:.2f}%, σ={msr_sigma*100:.2f}%, SR={sharpe_20k[msr_idx]:.3f}")

In [ ]:
# ── Q15: Full Figure with two subplots ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Portfolio Theory — Week 1 Visualisations', fontsize=15, fontweight='bold', y=1.01)

# ── SUBPLOT 1: Efficient Frontier Scatter ────────────────────────────────────
sc = ax1.scatter(
    sigma_p_20k * 100,
    E_Rp_20k    * 100,
    c=sharpe_20k,
    cmap='viridis',
    alpha=0.5,
    s=8,
    linewidths=0
)

# Colorbar
cbar = fig.colorbar(sc, ax=ax1, pad=0.02)
cbar.set_label('Sharpe Ratio', fontsize=11)

# Maximum Sharpe portfolio — gold star
ax1.scatter(msr_sigma * 100, msr_ret * 100,
            marker='*', s=250, color='gold', edgecolors='black',
            linewidths=0.8, zorder=5, label='Max Sharpe Portfolio')

# Individual assets — large dark circles
asset_labels  = ['Asset 1', 'Asset 2', 'Asset 3']
asset_sigmas  = sigma * 100     # individual σ in %
asset_returns = mu_assets * 100 # individual µ in %
asset_colors  = ['#1a1a2e', '#16213e', '#0f3460']

for i, (lbl, xs, yr) in enumerate(zip(asset_labels, asset_sigmas, asset_returns)):
    ax1.scatter(xs, yr, s=150, color=asset_colors[i],
                edgecolors='white', linewidths=0.8,
                zorder=6, label=lbl)
    ax1.annotate(lbl, xy=(xs, yr),
                 xytext=(xs + 0.3, yr + 0.2),
                 fontsize=9, color=asset_colors[i], fontweight='bold')

ax1.set_xlabel('Portfolio Risk σ_p (%)', fontsize=11)
ax1.set_ylabel('Expected Return E[R_p] (%)', fontsize=11)
ax1.set_title('Efficient Frontier (3-Asset Universe)', fontsize=12)
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.tick_params(labelsize=9)

# ── SUBPLOT 2: Correlation Sensitivity ───────────────────────────────────────
ax2.plot(rho_vals, sigma_p_rho * 100,
         color='royalblue', linewidth=2.5, label=r'$\sigma_p$ vs $\rho$')

# Weighted average risk dashed line
ax2.axhline(weighted_avg_risk * 100, color='crimson',
            linestyle='--', linewidth=1.8, label='Weighted Avg. Risk')

# Shade diversification benefit region
ax2.fill_between(
    rho_vals,
    sigma_p_rho * 100,
    weighted_avg_risk * 100,
    where=(sigma_p_rho * 100 < weighted_avg_risk * 100),
    color='limegreen',
    alpha=0.25,
    label='Diversification Benefit'
)

ax2.set_xlabel(r'Correlation $\rho$', fontsize=11)
ax2.set_ylabel('Portfolio Risk σ_p (%)', fontsize=11)
ax2.set_title('Correlation Sensitivity of Portfolio Risk', fontsize=12)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.tick_params(labelsize=9)
ax2.set_xlim(-1, 1)

plt.tight_layout()
plt.savefig('week1_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as 'week1_plots.png' at 150 dpi.")